# FUNGI v6 -- Pruning Pipeline
**Functional Unravelling of Network Geometry for Inference**

This notebook implements the complete FUNGI v6 pruning pipeline, upgraded from v5
with three major architectural changes:

1. **Data-Driven Diagnostics (Phase 0):** Replaces the fixed archetype weight system
   (Generalist/Structuralist/Modularist/Hierarchist) with dynamically computed
   utopian bounds and loss weights derived from the training h5ad file.

2. **Expanded Search Architecture (Phases 3-5):** Migrates from 12k LHS + joblib to
   100k Sobol + Ray Core for Phase 1, K-Means spatial niching for anchor extraction,
   and Trust Region Bayesian Optimization (TuRBO) for Phase 2 refinement.

3. **Expanded Shatter Criteria:** Adds spectral dominance ratio, Moran's I spatial
   coherence, GWCC percolation, and tighter alpha/clustering bounds to the fail-fast
   triage, alongside persistent homology and EPR checks in the audit phase.

The pipeline architecture: a YAML config file coordinates the run, src/ modules
handle the heavy computation, and this notebook orchestrates the phases.

## Phase 1 -- Environment Setup

### 1.1 Load Configuration

In [ ]:
import os
import yaml
from pathlib import Path

CONFIG_PATH = Path("fungi_config.yaml")
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

# Enforce single-threaded BLAS before any numerical import
if cfg["runtime"].get("single_threaded_blas", True):
    for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
        os.environ[var] = "1"

# Resolve paths
RAW_GRAPH_PATH = Path(cfg["input"]["graph_path"])
SC_DATA_PATH = Path(cfg["input"]["sc_data_path"]) if cfg["input"]["sc_data_path"] else None
OUTPUT_ROOT = Path(cfg["output"]["root_dir"])
SRC_ROOT = Path("src")

# Create output directories
for phase in cfg["output"]["phases"]:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)
Path(cfg["output"]["figures_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded from {CONFIG_PATH}")
print(f"Output root: {OUTPUT_ROOT}")

### 1.2 Library Imports

In [ ]:
import gc
import sys
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import scanpy as sc
import anndata as ad

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Core libraries imported.")
print(f"src/ directory: {SRC_ROOT.resolve()}")

### 1.3 Graph Ingestion

In [ ]:
from graph_utils import load_graph

raw_G, raw_sparse_mat = load_graph(RAW_GRAPH_PATH)

N_GENES = raw_sparse_mat.shape[0]
print(f"Graph loaded: {N_GENES:,} nodes, {raw_sparse_mat.nnz:,} edges")
print(f"Starting density: {nx.density(raw_G):.4%}")

## Phase 0 -- Data-Driven Diagnostic Calibration

This phase replaces the fixed archetype system entirely. It scans the training
h5ad to derive custom utopian bounds and loss weights for each of the six output
topology parameters (alpha, Gini, S_max, Q, C, rho).

The diagnostics rely on the principle that the topological structure of a causal
gene regulatory network directly dictates the statistical distribution of its
downstream perturbation effects. By analyzing the perturbation response data,
we reverse-engineer what the pruned graph topology should look like.

In [ ]:
from diagnostics import run_diagnostics

# Load the training h5ad
adata = sc.read_h5ad(cfg["input"]["sc_data_path"])
print(f"Training data: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")

# Run the full diagnostic pipeline
utopian_bounds, loss_weights, diagnostic_report = run_diagnostics(
    adata=adata,
    n_genes=N_GENES,
    cfg_diagnostics=cfg["diagnostics"],
    cfg_input=cfg["input"],
)

# Save diagnostics
diag_dir = OUTPUT_ROOT / "phase0_diagnostics"
pd.DataFrame([diagnostic_report]).to_json(diag_dir / "diagnostic_report.json", indent=2)
print("Diagnostic report saved.")

## Phase 2 -- Normalization

### 2.1 Adaptive Thresholding and Rank Standardization

In [ ]:
from scipy.stats import rankdata
from filtering import adaptive_threshold_filter

target_density = cfg["prefilter"]["target_density"]

# Adaptive threshold to reduce to the candidate pool
G_work = adaptive_threshold_filter(raw_sparse_mat, target_density=target_density)
gc.collect()

# Extract edge data from the filtered graph
G_work_coo = G_work.tocoo()
sources_raw = G_work_coo.row.copy()
targets_raw = G_work_coo.col.copy()
weights_raw = G_work_coo.data.copy()

# Rank-standardize weights (uniform distribution for DASH)
W_arr = rankdata(weights_raw, method="average").astype(np.float64) / len(weights_raw)

# Log-normalize out-degrees (compress scale-free hubs)
out_deg_raw = np.bincount(sources_raw, minlength=N_GENES).astype(np.float64)
D_arr = np.log1p(out_deg_raw)[sources_raw]

sources_arr = sources_raw
targets_arr = targets_raw

# Build candidate DataFrame for reference
candidate_df = pd.DataFrame({
    "source": sources_arr, "target": targets_arr,
    "W_raw": weights_raw, "W_norm": W_arr, "D_norm": D_arr,
})

print(f"Candidate pool: {len(candidate_df):,} edges")
print(f"W_norm range: [{W_arr.min():.4f}, {W_arr.max():.4f}]")
print(f"D_norm range: [{D_arr.min():.4f}, {D_arr.max():.4f}]")

### 2.2 Normalization Diagnostic Report

In [ ]:
total_possible = N_GENES * N_GENES
original_edges = raw_sparse_mat.nnz
work_edges = len(candidate_df)

report = f"""
{'=' * 72}
FUNGI v6 -- Phase 2: Normalization Report
{'=' * 72}
  Nodes (genes):         {N_GENES:,}
  Possible edges:        {total_possible:,}

  Original dense graph
    Edges:               {original_edges:,}
    Density:             {original_edges / total_possible:.4%}

  Filtered working graph
    Edges:               {work_edges:,}
    Density:             {work_edges / total_possible:.4%}
    Reduction:           {(1 - work_edges / original_edges) * 100:.1f}%
{'=' * 72}
"""
print(report)

## Phase 3 -- Expansive Search (Sobol + Ray)

### 3.1 Sobol Sequence Generation

In [ ]:
from search import generate_sobol_samples, get_static_bounds

es_cfg = cfg["expansive_search"]
hp_cfg = cfg["hyperparameter_bounds"]

sobol_params, lower_bounds, upper_bounds = generate_sobol_samples(
    n_genes=N_GENES,
    n_samples=es_cfg["n_samples"],
    hp_cfg=hp_cfg,
    seed=es_cfg["random_seed"],
)

# Extract perturbation targets for local-connectivity guardrail
perturbed_nodes = np.array([], dtype=int)

if SC_DATA_PATH is not None:
    pert_col = cfg["input"]["perturbation_column"]
    ctrl_label = cfg["input"]["control_label"]
    pert_genes = [
        g for g in adata.obs[pert_col].unique()
        if g != ctrl_label
    ]
    gene_list = list(adata.var_names)
    perturbed_nodes = np.array([
        gene_list.index(g) for g in pert_genes
        if g in gene_list and gene_list.index(g) < N_GENES
    ], dtype=int)
    print(f"Perturbation targets mapped: {len(perturbed_nodes):,} genes")

print(f"Sobol search space: {len(sobol_params):,} coordinates in 6D")

### 3.2 Expansive Search -- Batch Execution

In [ ]:
from search import execute_search_ray

t0 = time.time()

df_expansive = execute_search_ray(
    param_list=sobol_params,
    W_arr=W_arr,
    D_arr=D_arr,
    sources_arr=sources_arr,
    targets_arr=targets_arr,
    n_genes=N_GENES,
    perturbed_nodes=perturbed_nodes,
    utopian_bounds=utopian_bounds,
    loss_weights=loss_weights,
    shatter_cfg=cfg["shatter"],
    n_workers=es_cfg["n_workers"],
    chunk_size=es_cfg["chunk_size"],
)

elapsed = time.time() - t0
print(f"Expansive search complete: {len(df_expansive):,} graphs in {elapsed:.1f}s")

# Save results
phase3_dir = OUTPUT_ROOT / "phase3_expansive_search"
df_expansive.to_csv(phase3_dir / "expansive_search_results.csv", index=False)

### 3.3 Expansive Search -- Diagnostic Report

In [ ]:
df_viable = df_expansive[df_expansive["is_shattered"] == 0].copy()
df_shattered = df_expansive[df_expansive["is_shattered"] == 1]

n_viable = len(df_viable)
n_shattered = len(df_shattered)
shatter_rate = n_shattered / len(df_expansive) * 100

print(f"Results: {n_viable:,} viable | {n_shattered:,} shattered ({shatter_rate:.1f}%)")

if n_shattered > 0 and "shatter_reason" in df_shattered.columns:
    reason_counts = df_shattered["shatter_reason"].value_counts()
    print("
Shatter breakdown:")
    for reason, count in reason_counts.items():
        print(f"  {reason}: {count:,} ({count / n_shattered * 100:.1f}%)")

if n_viable > 0:
    print(f"
Viable graph statistics:")
    print(f"  Utopia loss: [{df_viable['utopia_loss'].min():.4f}, {df_viable['utopia_loss'].max():.4f}]")
    print(f"  Best 5 losses: {df_viable.nsmallest(5, 'utopia_loss')['utopia_loss'].values}")

## Phase 4 -- Spatial Niching (Bridge to TuRBO)

In [ ]:
from niching import extract_anchors

nich_cfg = cfg["niching"]

anchor_coords, anchor_losses, cluster_summary = extract_anchors(
    df_results=df_expansive,
    top_fraction=nich_cfg["top_fraction"],
    n_clusters=nich_cfg["n_clusters"],
    random_seed=cfg["runtime"]["random_seed"],
)

# Save
phase4_dir = OUTPUT_ROOT / "phase4_niching"
np.save(phase4_dir / "anchor_coords.npy", anchor_coords)
np.save(phase4_dir / "anchor_losses.npy", anchor_losses)
cluster_summary.to_csv(phase4_dir / "cluster_summary.csv", index=False)

print(f"Anchors saved: {len(anchor_coords)} coordinates")

## Phase 5 -- TuRBO Refinement

Trust Region Bayesian Optimization replaces the v5 GMM + RF surrogate.
Each of the 50 anchor coordinates seeds an independent trust region with its
own local Gaussian Process surrogate. Regions adaptively expand on success
and contract on failure, converging to the mathematical minimum of the
utopian loss landscape within their local neighborhood.

In [ ]:
from turbo_search import run_turbo_refinement
from search import execute_search_ray

def turbo_evaluate_fn(batch_params):
    """Wraps the Ray executor for TuRBO candidate evaluation."""
    df = execute_search_ray(
        param_list=batch_params,
        W_arr=W_arr, D_arr=D_arr,
        sources_arr=sources_arr, targets_arr=targets_arr,
        n_genes=N_GENES, perturbed_nodes=perturbed_nodes,
        utopian_bounds=utopian_bounds, loss_weights=loss_weights,
        shatter_cfg=cfg["shatter"],
        n_workers=es_cfg["n_workers"],
        chunk_size=min(es_cfg["chunk_size"], len(batch_params)),
    )
    return df.to_dict("records")

t0 = time.time()

df_turbo, best_result = run_turbo_refinement(
    anchor_coords=anchor_coords,
    anchor_losses=anchor_losses,
    lower=lower_bounds,
    upper=upper_bounds,
    evaluate_fn=turbo_evaluate_fn,
    turbo_cfg=cfg["turbo"],
)

elapsed = time.time() - t0
print(f"TuRBO complete: {len(df_turbo):,} evaluations in {elapsed:.1f}s")

# Save
phase5_dir = OUTPUT_ROOT / "phase5_turbo_refinement"
df_turbo.to_csv(phase5_dir / "turbo_results.csv", index=False)

## Phase 5b -- Merge and Audit Cohort Extraction

In [ ]:
# Merge all results
df_all = pd.concat([df_expansive, df_turbo], ignore_index=True)
df_all_viable = df_all[df_all["is_shattered"] == 0].copy()
df_all_viable = df_all_viable.sort_values("utopia_loss").reset_index(drop=True)

cohort_size = cfg["audit"]["cohort_size"]
df_audit = df_all_viable.head(cohort_size).copy()

print(f"Total viable graphs: {len(df_all_viable):,}")
print(f"Audit cohort: top {cohort_size} by utopian loss")
print(f"  Loss range: [{df_audit['utopia_loss'].min():.4f}, {df_audit['utopia_loss'].max():.4f}]")

# Save
df_audit.to_csv(OUTPUT_ROOT / "phase6_audit" / "audit_cohort.csv", index=False)

## Phase 6 -- Biological Audit

The audit phase runs expensive topology checks (persistent homology, EPR)
that are too slow for the 100k expansive search. These are computed only on
the final cohort.

In [ ]:
audit_cfg = cfg["audit"]

if audit_cfg.get("persistence_homology", False):
    from topology import compute_persistence_wasserstein
    print("Computing persistent homology for audit cohort...")
    # This would iterate over the cohort and compute Wasserstein distances
    # against the dense parent graph. Implementation depends on the
    # specific graph reconstruction logic.
    print("  (Persistence homology computation placeholder -- requires")
    print("   graph reconstruction from hyperparameters for each cohort member.)")

if audit_cfg.get("epr_check", False):
    from topology import compute_epr
    print("Computing EPR for audit cohort...")
    print("  (EPR computation placeholder -- requires ground truth edge set")
    print("   derived from perturbation data.)")

print("
Audit phase complete. Cohort exported.")
print(f"Final cohort: {len(df_audit)} candidate graphs ready for downstream evaluation.")

## Phase 7 -- Export Pipeline Artifacts

In [ ]:
import joblib as jl

artifact_path = OUTPUT_ROOT / "phase2_normalization" / "pipeline_artifacts.joblib"

artifacts = {
    "candidate_df": candidate_df,
    "N_GENES": N_GENES,
    "utopian_bounds": utopian_bounds,
    "loss_weights": loss_weights,
    "diagnostic_report": diagnostic_report,
}

jl.dump(artifacts, artifact_path)
print(f"Pipeline artifacts exported to {artifact_path.name}")
print("FUNGI v6 pipeline complete.")